In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
# 中文字体&负号 修复（Windows）
from pathlib import Path
from matplotlib import rcParams, font_manager
import matplotlib, shutil
print(matplotlib.get_cachedir())  # 看一下缓存目录
# 然后手动删除该目录下的 fontlist-v***.json（或直接删整个缓存目录）

# 1) 把常见中文字体注册进 Matplotlib
candidates = [
    r"C:\Windows\Fonts\msyh.ttc",     # 微软雅黑
    r"C:\Windows\Fonts\simhei.ttf",   # 黑体
    r"C:\Windows\Fonts\msyhbd.ttc",   # 微软雅黑 Bold
]
for p in candidates:
    if Path(p).exists():
        font_manager.fontManager.addfont(p)

# 2) 设为默认字体 + 修复负号（unicode minus）
rcParams["font.family"] = "sans-serif"
rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
rcParams["axes.unicode_minus"] = False  # 避免负号显示成方块

# 可选：如果你把 Noto CJK 放到项目里（跨平台更稳）
# font_manager.fontManager.addfont("fonts/NotoSansCJKsc-Regular.otf")
# rcParams["font.family"] = "Noto Sans CJK SC"

# ========= 配置（按需改） =========
F15 = Path("btc_2015.xlsx")
F21 = Path("btc_2021.xlsx")
F25 = Path("btc_2025.xlsx")
OUTDIR = Path("btc_fit_outputs09142025"); OUTDIR.mkdir(exist_ok=True)

# 参考窗口：用 2024-06-01 → 2025-08-15 作为 2025 的“锚点区间”
REF_START_25 = pd.Timestamp("2024-06-01")
ANCHOR_25    = pd.Timestamp("2025-08-15")   # 2025 假定峰值
POST_DAYS    = 60                           # 峰值后的展示天数（25年会被数据上限截断）
# 2017/2021 的峰值（2017 固定；2021 若想固定也可在此填日期）
PEAK_2017 = pd.Timestamp("2017-12-19")
PEAK_2021 = None   # 比如 pd.Timestamp("2021-11-10")；None=自动取 2021 年内最高日

# —— 波动退火：把旧周期的幅度缩到与 2025 更接近（只改幅度，不改形状）
# 你可以用“手工等级”或“历史标准差”或二者合成（hybrid）
VOL_LEVEL_2017, VOL_LEVEL_2021, VOL_LEVEL_2025 = 15.0, 7.0, 1.5
VOL_ALPHA = 0.5           # =0.5 表示“开方”；=1 表示线性比；可自行调整
SCALE_METHOD = "manual"   # 选 "manual" / "std" / "hybrid"
PRE_STD_SPAN = 90         # 用峰值前多少天估算历史 std（method 含 std/hybrid 时生效）

# ========= 工具 =========
def read_two_col_excel(fp: Path) -> pd.DataFrame:
    """读两列（时间、价格），兼容“横着两行”，去重、按时间升序。"""
    assert fp.exists(), f"未找到文件：{fp}"
    df_raw = pd.read_excel(fp, engine="openpyxl", header=None)
    if df_raw.shape[0] <= 3 and df_raw.shape[1] > df_raw.shape[0]:
        df_raw = df_raw.T
    df = df_raw.iloc[:, :2].copy()
    df.columns = ["ts", "price"]
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df = df.dropna(subset=["ts","price"]).sort_values("ts").drop_duplicates(subset=["ts"], keep="last")
    return df

def to_daily(series_df: pd.DataFrame) -> pd.Series:
    """转日频收盘(last)+前向填充，得到连续日序列。"""
    s = series_df.set_index("ts")["price"].resample("D").last()
    s = s.reindex(pd.date_range(s.index.min().normalize(), s.index.max().normalize(), freq="D")).ffill()
    s.index.name = "ts"
    return s

def window_by_anchor(series: pd.Series, anchor_date: pd.Timestamp, pre_days: int, post_days: int) -> pd.Series:
    """以 anchor 为中心，取前 pre_days、后 post_days 的窗口；若右端超出数据就自动截断。"""
    anchor_date = pd.Timestamp(anchor_date).normalize()
    full_idx = pd.date_range(series.index.min(), max(series.index.max(), anchor_date), freq="D")
    s = series.reindex(full_idx).ffill()
    lo = max(anchor_date - pd.Timedelta(days=pre_days), s.index.min())
    hi = min(anchor_date + pd.Timedelta(days=post_days), s.index.max())
    return s.loc[lo:hi]

def pct_curve(series: pd.Series, anchor_date: pd.Timestamp) -> pd.DataFrame:
    """相对锚点百分比曲线（锚点=0%），index=日期，含 rel_day & dd_pct。"""
    anchor_date = pd.Timestamp(anchor_date).normalize()
    anchor_px = float(series.loc[anchor_date] if anchor_date in series.index else series.loc[:anchor_date].iloc[-1])
    rel_day = (series.index - anchor_date).days
    dd_pct = (series / anchor_px - 1.0) * 100.0
    return pd.DataFrame({"rel_day": rel_day, "dd_pct": dd_pct.values}, index=series.index), anchor_px

def fit_metrics(a: pd.Series, b: pd.Series):
    idx = a.index.intersection(b.index)
    if len(idx) < 3: return np.nan, np.nan
    A, B = a.loc[idx].values, b.loc[idx].values
    r = float(np.corrcoef(A, B)[0,1]); rmse = float(np.sqrt(np.mean((A-B)**2)))
    return r, rmse

def scale_factor(std_old, std_new, level_old, level_new, method="manual", alpha=0.5):
    """
    计算缩放系数（乘到旧周期上）：
      - manual : (level_new / level_old) ** alpha     # 你给的“9→3→1 等级”，alpha=0.5=开方
      - std    : std_new / std_old                    # 用历史标准差比例
      - hybrid : (std_new / std_old) * (level_new / level_old) ** alpha
    """
    if method == "manual":
        return (level_new / level_old) ** alpha
    elif method == "std":
        return (std_new / std_old) if std_old and std_old > 0 else 1.0
    else:  # hybrid
        left  = (std_new / std_old) if std_old and std_old > 0 else 1.0
        right = (level_new / level_old) ** alpha
        return left * right

# ========= 读三表 → 日频 =========
s15 = to_daily(read_two_col_excel(F15))
s21 = to_daily(read_two_col_excel(F21))
s25 = to_daily(read_two_col_excel(F25))

# ========= 2025 参考窗（含峰后 POST_DAYS）=========
pre_len = (ANCHOR_25 - REF_START_25).days
win25 = window_by_anchor(s25, ANCHOR_25, pre_len, POST_DAYS)
curve25, anchor25 = pct_curve(win25, ANCHOR_25)
c25 = curve25.set_index("rel_day")["dd_pct"]

# ========= 2021：峰值（自动或固定）→ 同长度窗口（含峰后）=========
s21_y = s21.loc["2021-01-01":"2021-12-31"]
assert not s21_y.empty, "2021 年数据缺失，请检查 btc_2021.xlsx"
peak21 = (PEAK_2021 if PEAK_2021 is not None else s21_y.idxmax())
win21 = window_by_anchor(s21, peak21, pre_len, POST_DAYS)
curve21, anchor21 = pct_curve(win21, peak21)
c21 = curve21.set_index("rel_day")["dd_pct"]

# ========= 2017：峰值固定 11/30 → 同长度窗口（含峰后）=========
s17_y = s15.loc["2017-01-01":"2017-12-31"]
assert not s17_y.empty, "2017 年数据缺失，请检查 btc_2015.xlsx 是否覆盖到 2017"
peak17 = PEAK_2017
win17 = window_by_anchor(s15, peak17, pre_len, POST_DAYS)
curve17, anchor17 = pct_curve(win17, peak17)
c17 = curve17.set_index("rel_day")["dd_pct"]

# ========= 预估 std（用于 std/hybrid）——用峰前 PRE_STD_SPAN 天 =========
def pre_std(c):
    idx = [d for d in range(-min(PRE_STD_SPAN, pre_len), 1) if d in c.index]
    return c.loc[idx].std(ddof=0)

std25 = pre_std(c25); std21 = pre_std(c21); std17 = pre_std(c17)

# ========= 计算缩放系数（把 2017/2021 缩成接近 2025）=========
scale21 = scale_factor(std21, std25, VOL_LEVEL_2021, VOL_LEVEL_2025, method=SCALE_METHOD, alpha=VOL_ALPHA)
scale17 = scale_factor(std17, std25, VOL_LEVEL_2017, VOL_LEVEL_2025, method=SCALE_METHOD, alpha=VOL_ALPHA)
c21_s, c17_s = c21 * scale21, c17 * scale17

# ========= 拟合指标（未缩放 & 缩放后；峰前/峰后 分开看）=========
def split_metrics(c_old, name):
    pre_idx  = [i for i in c25.index if i <= 0 and i in c_old.index]
    post_idx = [i for i in c25.index if i >= 0 and i in c_old.index]
    r_pre, rmse_pre   = fit_metrics(c25.loc[pre_idx],  c_old.loc[pre_idx])
    r_post, rmse_post = fit_metrics(c25.loc[post_idx], c_old.loc[post_idx])
    return r_pre, rmse_pre, r_post, rmse_post

m21_raw  = split_metrics(c21,  "2021_raw")
m17_raw  = split_metrics(c17,  "2017_raw")
m21_scal = split_metrics(c21_s,"2021_scaled")
m17_scal = split_metrics(c17_s,"2017_scaled")

# ========= 保存合并数据 =========
merged = (pd.DataFrame({"dd_pct_2025": c25})
          .join(pd.DataFrame({"dd_pct_2021": c21}), how="outer")
          .join(pd.DataFrame({"dd_pct_2017": c17}), how="outer"))
merged["dd_pct_2021_scaled"] = merged["dd_pct_2021"] * scale21
merged["dd_pct_2017_scaled"] = merged["dd_pct_2017"] * scale17
merged.index.name = "rel_day"
merged.to_csv(OUTDIR / "fit_2017_2021_vs_2025_window_with_tail.csv", encoding="utf-8-sig")

with open(OUTDIR / "fit_metrics_with_tail.txt","w",encoding="utf-8") as f:
    f.write(f"窗口（2025）：{REF_START_25.date()} → {ANCHOR_25.date()}，峰后展示 {POST_DAYS} 天\n")
    f.write(f"峰值：2017={peak17.date()}，2021={peak21.date()}，2025={ANCHOR_25.date()}\n")
    f.write(f"锚点价：2017={anchor17:,.2f}，2021={anchor21:,.2f}，2025={anchor25:,.2f}\n")
    f.write(f"std(峰前)：2017={std17:.3f}，2021={std21:.3f}，2025={std25:.3f}\n")
    f.write(f"scale(method={SCALE_METHOD}, alpha={VOL_ALPHA}): 2017→25 = {scale17:.3f}，2021→25 = {scale21:.3f}\n\n")
    f.write("—— 未缩放 ——\n")
    f.write(f"2021: r_pre={m21_raw[0]:.4f}, rmse_pre={m21_raw[1]:.3f} | r_post={m21_raw[2]:.4f}, rmse_post={m21_raw[3]:.3f}\n")
    f.write(f"2017: r_pre={m17_raw[0]:.4f}, rmse_pre={m17_raw[1]:.3f} | r_post={m17_raw[2]:.4f}, rmse_post={m17_raw[3]:.3f}\n\n")
    f.write("—— 缩放后（波动退火）——\n")
    f.write(f"2021_scaled: r_pre={m21_scal[0]:.4f}, rmse_pre={m21_scal[1]:.3f} | r_post={m21_scal[2]:.4f}, rmse_post={m21_scal[3]:.3f}\n")
    f.write(f"2017_scaled: r_pre={m17_scal[0]:.4f}, rmse_pre={m17_scal[1]:.3f} | r_post={m17_scal[2]:.4f}, rmse_post={m17_scal[3]:.3f}\n")

# ========= 画图（保存）=========
# 图1：波动退火（与你发的图类似，但含峰后）
plt.figure(figsize=(12,4.5))
plt.plot(c25.index, c25.values, label="2025 参考窗")
plt.plot(c21_s.index, c21_s.values, label=f"2021（缩放 ×{scale21:.2f}）")
plt.plot(c17_s.index, c17_s.values, label=f"2017（缩放 ×{scale17:.2f}）")
plt.axvline(0, linestyle="--")
plt.xlabel("相对天数（锚点=0）"); plt.ylabel("相对锚点涨跌（%）")
plt.title("锚点归一化百分比：波动退火后，与 2025 对齐（含峰后）")
plt.legend(); plt.tight_layout()
plt.savefig(OUTDIR / "fit_scaled_with_tail.png", dpi=170); plt.close()

# 图2：未缩放，对比真实幅度（方便直观看到 2017≫2021≫2025 的波动差）
plt.figure(figsize=(12,4.5))
plt.plot(c25.index, c25.values, label="2025（原幅度）")
plt.plot(c21.index, c21.values, label="2021（原幅度）")
plt.plot(c17.index, c17.values, label="2017（原幅度）")
plt.axvline(0, linestyle="--")
plt.xlabel("相对天数（锚点=0）"); plt.ylabel("相对锚点涨跌（%）")
plt.title("锚点归一化百分比：原始幅度（含峰后）")
plt.legend(); plt.tight_layout()
plt.savefig(OUTDIR / "fit_unscaled_with_tail0588v2.png", dpi=170); plt.close()

print("已输出到：", OUTDIR.resolve())



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ===== 配置 =====
FILES = [Path("btc_2015.xlsx"), Path("btc_2021.xlsx"), Path("btc_2025.xlsx")]
OUTDIR = Path("btc_fit_outputs09142025"); OUTDIR.mkdir(exist_ok=True)

# AHR999 参数（与原 Pine Script 一致）
LEN_GMA = 200                 # 200日几何均值
K, B = 5.84, -17.01           # 估值中枢：10^(K * log10(age) + B)
BITCOIN_BIRTH = pd.Timestamp("2009-01-03")  # 比特币生日

# ===== 工具函数 =====
def read_two_col_excel(fp: Path) -> pd.DataFrame:
    """读取两列（时间、价格），支持‘横着两行’；返回按时间升序去重后的 DataFrame。"""
    assert fp.exists(), f"未找到文件：{fp}"
    df = pd.read_excel(fp, engine="openpyxl", header=None)
    if df.shape[0] <= 3 and df.shape[1] > df.shape[0]:
        df = df.T  # 横排两行 → 竖列
    df = df.iloc[:, :2].copy()
    df.columns = ["ts", "price"]
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    return df.dropna().drop_duplicates("ts").sort_values("ts")

def to_daily_close(df: pd.DataFrame) -> pd.Series:
    """转为日频‘收盘’（last），并前向填充补全日期。"""
    s = df.set_index("ts")["price"].resample("D").last()
    s = s.reindex(pd.date_range(s.index.min().normalize(), s.index.max().normalize(), freq="D")).ffill()
    s.index.name = "date"
    return s

# ===== 1) 读取三个文件 → 合并成一条日线价格 =====
daily_series = []
for f in FILES:
    daily_series.append(to_daily_close(read_two_col_excel(f)))

# 合并（重叠日期取首个非空；理论上三表在重叠处价格相同）
df_all = pd.concat(daily_series, axis=1)
price = df_all.bfill(axis=1).iloc[:, 0].rename("price")  # 取每行第一个非空
price = price.sort_index()
assert not price.empty, "价格序列为空，请检查输入文件"

# ===== 2) 计算 AHR999 各组成项 =====
out = pd.DataFrame(index=price.index)
out["price"] = price.astype(float)

# 200日‘几何均值’（避免乘法溢出，用 log 的滚动平均）
logp = np.log(out["price"])
gma200 = np.exp(logp.rolling(LEN_GMA, min_periods=LEN_GMA).mean())
out["gma200"] = gma200

# 平均成本因子：avg_index = P / GMA200
out["avg_index"] = out["price"] / out["gma200"]



# 年龄估值中枢：estimate_price = 10^(K*log10(age)+B)
# 用“自 2009-01-03 起的天数”作为 age（浮点），并把非正数设为 NaN，避免 log10 出错
age_days = pd.Series(
    ((out.index - BITCOIN_BIRTH) / pd.Timedelta(days=1)).astype(float),
    index=out.index,
    dtype=float
)
age_days = age_days.where(age_days > 0, np.nan)

with np.errstate(divide="ignore", invalid="ignore"):
    estimate_price = np.power(10.0, K * np.log10(age_days) + B)

out["estimate_price"] = estimate_price
out["estimate_index"] = out["price"] / out["estimate_price"]
out["ahr999"] = out["avg_index"] * out["estimate_index"]




# ===== 3) 导出 =====
xlsx_path = OUTDIR / "ahr999_daily.xlsx"
csv_path  = OUTDIR / "ahr999_daily.csv"

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as w:
    out.to_excel(w, sheet_name="AHR999")

out.to_csv(csv_path, index_label="date", encoding="utf-8-sig")

print("已保存：", xlsx_path.resolve())
print("也导出：", csv_path.resolve())
print(out.tail(5))


In [ ]:
# step_fit_ahr999_peak_alignment_optimized.py
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager

# ====== 中文字体（可选）======
for p in [r"C:\Windows\Fonts\msyh.ttc", r"C:\Windows\Fonts\simhei.ttf"]:
    if Path(p).exists():
        font_manager.fontManager.addfont(p)
rcParams["font.family"] = "sans-serif"
rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
rcParams["axes.unicode_minus"] = False

# ========= 文件 & 目录 =========
F15 = Path("btc_2015.xlsx")
F21 = Path("btc_2021.xlsx")
F25 = Path("btc_2025.xlsx")
OUTDIR = Path("btc_fit_outputs09142025"); OUTDIR.mkdir(exist_ok=True)

# ========= AHR999 参数 =========
LEN_GMA = 200
K, B = 5.84, -17.01
BITCOIN_BIRTH = pd.Timestamp("2009-01-03")

# ========= 锚点窗口参数（沿用你的逻辑）=========
REF_START_25 = pd.Timestamp("2024-06-01")
ANCHOR_25    = pd.Timestamp("2025-08-15")
POST_DAYS    = 60
PEAK_2017    = pd.Timestamp("2017-12-19")
PEAK_2021    = None  # None=自动取 2021 年内最高价

# ========= 新增：拟合控制 =========
SMOOTH_WIN = 7               # 滑动平均窗口（天）；0=关闭
FIT_SIDE   = "pre"           # "pre" 只用峰前拟合 γ 与 a；"both" 用全窗口
WEIGHT_TAU = 60              # 权重衰减尺度（天），越小越重视靠近 0 天
SEPARATE_POST_SCALE = True   # 峰后是否单独估一个 a_post
GAMMA_MIN, GAMMA_MAX, GAMMA_N = 0.75, 1.30, 56  # γ 搜索范围与步数

# ========= 工具 =========
def read_two_col_excel(fp: Path) -> pd.DataFrame:
    assert fp.exists(), f"未找到文件：{fp}"
    df = pd.read_excel(fp, engine="openpyxl", header=None)
    if df.shape[0] <= 3 and df.shape[1] > df.shape[0]:
        df = df.T
    df = df.iloc[:, :2].copy()
    df.columns = ["ts", "price"]
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    return df.dropna().sort_values("ts").drop_duplicates("ts")

def to_daily(df: pd.DataFrame) -> pd.Series:
    s = df.set_index("ts")["price"].resample("D").last()
    s = s.reindex(pd.date_range(s.index.min().normalize(), s.index.max().normalize(), freq="D")).ffill()
    s.index.name = "ts"
    return s

def window_by_anchor(series: pd.Series, anchor_date, pre_days, post_days):
    anchor_date = pd.Timestamp(anchor_date).normalize()
    full_idx = pd.date_range(series.index.min(), max(series.index.max(), anchor_date), freq="D")
    s = series.reindex(full_idx).ffill()
    lo = max(anchor_date - pd.Timedelta(days=pre_days), s.index.min())
    hi = min(anchor_date + pd.Timedelta(days=post_days), s.index.max())
    return s.loc[lo:hi]

def pct_curve(series: pd.Series, anchor_date):
    anchor_date = pd.Timestamp(anchor_date).normalize()
    anchor_v = float(series.loc[anchor_date] if anchor_date in series.index else series.loc[:anchor_date].iloc[-1])
    rel_day = (series.index - anchor_date).days
    dd_pct  = (series / anchor_v - 1.0) * 100.0
    return pd.DataFrame({"rel_day": rel_day, "dd_pct": dd_pct.values}, index=series.index), anchor_v

def ahr999_from_price(price: pd.Series) -> pd.Series:
    logp = np.log(price)
    gma200 = np.exp(logp.rolling(LEN_GMA, min_periods=LEN_GMA).mean())
    age_days = ((price.index - BITCOIN_BIRTH) / pd.Timedelta(days=1)).astype(float)
    age_days = pd.Series(age_days, index=price.index).where(lambda s: s > 0, np.nan)
    with np.errstate(divide="ignore", invalid="ignore"):
        estimate_price = np.power(10.0, K * np.log10(age_days) + B)
    ahr = (price / gma200) * (price / estimate_price)
    ahr.name = "ahr999"
    return ahr

def smooth(s: pd.Series, win: int) -> pd.Series:
    if win and win > 1:
        return s.rolling(win, center=True, min_periods=max(1, win//2)).mean()
    return s

def weight(days: np.ndarray, side: str, tau: float) -> np.ndarray:
    if side == "pre":
        m = days <= 0
        w = np.exp(-(np.abs(days) / tau))
        w[~m] = 0.0
        return w
    elif side == "post":
        m = days >= 0
        w = np.exp(-(np.abs(days) / tau))
        w[~m] = 0.0
        return w
    else:
        return np.exp(-(np.abs(days) / tau))

def interp_series(rel_days_src: np.ndarray, vals_src: np.ndarray,
                  rel_days_tgt: np.ndarray, gamma: float) -> np.ndarray:
    """在源曲线的相对天数轴上做 time-warp：取 src(d * gamma)。超出范围返回 NaN。"""
    x = rel_days_tgt * gamma
    lo, hi = rel_days_src[0], rel_days_src[-1]
    y = np.full_like(rel_days_tgt, np.nan, dtype=float)
    mask = (x >= lo) & (x <= hi)
    y[mask] = np.interp(x[mask], rel_days_src, vals_src)
    return y

def fit_scale(target: np.ndarray, source: np.ndarray, w: np.ndarray) -> float:
    m = (~np.isnan(target)) & (~np.isnan(source)) & (w > 0)
    if m.sum() < 3:
        return 1.0
    A = source[m]; Y = target[m]; W = w[m]
    num = np.sum(W * A * Y)
    den = np.sum(W * A * A)
    return float(num / den) if den > 1e-12 else 1.0

def rmse(target: np.ndarray, pred: np.ndarray, w: np.ndarray) -> float:
    m = (~np.isnan(target)) & (~np.isnan(pred)) & (w > 0)
    if m.sum() < 3:
        return np.nan
    err2 = ((pred[m] - target[m]) ** 2) * w[m]
    return float(np.sqrt(np.sum(err2) / np.sum(w[m])))

def optimize_gamma_and_scale(c_old: pd.Series, c_tgt: pd.Series,
                             side="pre", tau=60,
                             gmin=0.8, gmax=1.2, gn=41):
    """在给定 side（pre/both）上搜索 γ，并在每个 γ 下闭式解 a，最小化加权 RMSE。"""
    days_tgt = c_tgt.index.values.astype(float)
    y_tgt = c_tgt.values.astype(float)
    days_src = c_old.index.values.astype(float)
    x_src = c_old.values.astype(float)

    w = weight(days_tgt, side=("pre" if side=="pre" else "both"), tau=tau)
    gammas = np.linspace(gmin, gmax, gn)

    best = {"gamma": 1.0, "a": 1.0, "rmse": np.inf}
    for g in gammas:
        x_warp = interp_series(days_src, x_src, days_tgt, g)  # src(d * g)
        a = fit_scale(y_tgt, x_warp, w)                      # y ≈ a * x_warp
        pred = a * x_warp
        e = rmse(y_tgt, pred, w)
        if e < best["rmse"]:
            best = {"gamma": float(g), "a": float(a), "rmse": float(e)}
    return best

# ========= 读取并合并价格 =========
s15, s21, s25 = map(to_daily, map(read_two_col_excel, [F15, F21, F25]))
df_all = pd.concat([s15.rename("p15"), s21.rename("p21"), s25.rename("p25")], axis=1)
price = df_all.bfill(axis=1).iloc[:, 0].rename("price").astype(float).sort_index()
assert not price.empty, "价格序列为空"

# ========= 计算 AHR999 =========
ahr = ahr999_from_price(price)
ahr.to_csv(OUTDIR / "ahr999_daily_from_inputs.csv", index_label="date", encoding="utf-8-sig")

# ========= 锚点与窗口（按价格峰值日期）=========
pre_len = (ANCHOR_25 - REF_START_25).days
s21_y = s21.loc["2021-01-01":"2021-12-31"];  assert not s21_y.empty, "缺 2021"
peak21 = (PEAK_2021 if PEAK_2021 is not None else s21_y.idxmax())
peak17 = PEAK_2017
peak25 = ANCHOR_25

win25 = window_by_anchor(ahr, peak25, pre_len, POST_DAYS)
win21 = window_by_anchor(ahr, peak21, pre_len, POST_DAYS)
win17 = window_by_anchor(ahr, peak17, pre_len, POST_DAYS)

curve25, anchor25 = pct_curve(win25, peak25)
curve21, anchor21 = pct_curve(win21, peak21)
curve17, anchor17 = pct_curve(win17, peak17)

c25 = curve25.set_index("rel_day")["dd_pct"]
c21 = curve21.set_index("rel_day")["dd_pct"]
c17 = curve17.set_index("rel_day")["dd_pct"]

# 平滑
c25 = smooth(c25, SMOOTH_WIN)
c21 = smooth(c21, SMOOTH_WIN)
c17 = smooth(c17, SMOOTH_WIN)

# ========= 优化：γ + a（峰前）=========
opt21 = optimize_gamma_and_scale(c21, c25, side=("pre" if FIT_SIDE=="pre" else "both"),
                                 tau=WEIGHT_TAU, gmin=GAMMA_MIN, gmax=GAMMA_MAX, gn=GAMMA_N)
opt17 = optimize_gamma_and_scale(c17, c25, side=("pre" if FIT_SIDE=="pre" else "both"),
                                 tau=WEIGHT_TAU, gmin=GAMMA_MIN, gmax=GAMMA_MAX, gn=GAMMA_N)

# —— 用最优 γ 做全窗口插值，再做幅度缩放 —— #
def apply_timewarp_and_scale(c_old, gamma, a_pre, separate_post=False):
    days = c25.index.values.astype(float)
    days_src = c_old.index.values.astype(float)
    vals_src = c_old.values.astype(float)

    # 全窗口 time-warp
    warped = interp_series(days_src, vals_src, days, gamma)
    out_pre = pd.Series(warped, index=c25.index, dtype=float)

    # 峰前幅度 a_pre
    pre_mask = out_pre.index.values <= 0
    scaled = out_pre.copy()
    scaled[pre_mask] = scaled[pre_mask] * a_pre

    # 峰后可选单独 a_post（用同 γ，在 post 上再拟合一次）
    if separate_post:
        w_post = weight(days, "post", WEIGHT_TAU)
        a_post = fit_scale(c25.values, out_pre.values, w_post)
        scaled[~pre_mask] = scaled[~pre_mask] * a_post
        return scaled, a_post
    return scaled, None

c21_fit, a21_post = apply_timewarp_and_scale(c21, opt21["gamma"], opt21["a"], SEPARATE_POST_SCALE)
c17_fit, a17_post = apply_timewarp_and_scale(c17, opt17["gamma"], opt17["a"], SEPARATE_POST_SCALE)

# ========= 指标 =========
def split_metrics(a: pd.Series, b: pd.Series):
    pre = a.index[a.index <= 0]
    post = a.index[a.index >= 0]
    def _r_rmse(x, y):
        idx = x.dropna().index.intersection(y.dropna().index)
        if len(idx) < 3: return (np.nan, np.nan)
        X, Y = x.loc[idx].values, y.loc[idx].values
        r = float(np.corrcoef(X, Y)[0,1])
        rmse = float(np.sqrt(np.mean((X-Y)**2)))
        return r, rmse
    rp, ep = _r_rmse(c25.loc[pre],  b.loc[pre])
    ro, eo = _r_rmse(c25.loc[post], b.loc[post])
    return rp, ep, ro, eo

m21_fit = split_metrics(c25, c21_fit)
m17_fit = split_metrics(c25, c17_fit)

# ========= 导出 CSV =========
merged = pd.DataFrame({"dd_pct_ahr_2025": c25,
                       "dd_pct_ahr_2021": c21,
                       "dd_pct_ahr_2017": c17,
                       "dd_pct_ahr_2021_timewarp_scaled": c21_fit,
                       "dd_pct_ahr_2017_timewarp_scaled": c17_fit})
merged.index.name = "rel_day"
merged.to_csv(OUTDIR / "ahr999_fit_timewarp_scaled.csv", encoding="utf-8-sig")

# ========= 写出拟合参数 =========
with open(OUTDIR / "ahr999_fit_params_timewarp.txt","w",encoding="utf-8") as f:
    f.write(f"[AHR999 Time-warp 拟合]\n")
    f.write(f"窗口：{REF_START_25.date()} → {ANCHOR_25.date()}，峰后 {POST_DAYS} 天；平滑={SMOOTH_WIN}\n")
    f.write(f"权重：tau={WEIGHT_TAU}；γ∈[{GAMMA_MIN},{GAMMA_MAX}]×{GAMMA_N}；预/后单独scale={SEPARATE_POST_SCALE}\n\n")
    f.write(f"2021: gamma={opt21['gamma']:.4f}, a_pre={opt21['a']:.4f}, rmse(pre)={opt21['rmse']:.3f}")
    if a21_post is not None: f.write(f", a_post={a21_post:.4f}")
    f.write("\n")
    f.write(f"2017: gamma={opt17['gamma']:.4f}, a_pre={opt17['a']:.4f}, rmse(pre)={opt17['rmse']:.3f}")
    if a17_post is not None: f.write(f", a_post={a17_post:.4f}")
    f.write("\n\n")
    f.write("—— 与 2025 的相关/误差（% 单位）——\n")
    f.write(f"2021_fit: r_pre={m21_fit[0]:.4f}, rmse_pre={m21_fit[1]:.2f} | r_post={m21_fit[2]:.4f}, rmse_post={m21_fit[3]:.2f}\n")
    f.write(f"2017_fit: r_pre={m17_fit[0]:.4f}, rmse_pre={m17_fit[1]:.2f} | r_post={m17_fit[2]:.4f}, rmse_post={m17_fit[3]:.2f}\n")

# ========= 画图 =========
# 原始幅度（参考）
plt.figure(figsize=(12,4.5))
plt.plot(c25.index, c25.values, label="2025（AHR）")
plt.plot(c21.index, c21.values, label="2021（AHR）")
plt.plot(c17.index, c17.values, label="2017（AHR）")
plt.axvline(0, linestyle="--"); plt.xlabel("相对天数（锚点=0）"); plt.ylabel("AHR999 相对锚点（%）")
plt.title("AHR999：锚点归一化百分比（原始幅度，含峰后）")
plt.legend(); plt.tight_layout()
plt.savefig(OUTDIR / "ahr999_fit_unscaled_with_tail.png", dpi=170); plt.close()

# Time-warp + scale 后
plt.figure(figsize=(12,4.5))
plt.plot(c25.index, c25.values, label="2025 参考窗（AHR）")
plt.plot(c21_fit.index, c21_fit.values, label=f"2021（γ={opt21['gamma']:.2f}, a_pre={opt21['a']:.2f}" + (f", a_post={a21_post:.2f}" if a21_post is not None else "") + ")")
plt.plot(c17_fit.index, c17_fit.values, label=f"2017（γ={opt17['gamma']:.2f}, a_pre={opt17['a']:.2f}" + (f", a_post={a17_post:.2f}" if a17_post is not None else "") + ")")
plt.axvline(0, linestyle="--"); plt.xlabel("相对天数（锚点=0）"); plt.ylabel("AHR999 相对锚点（%）")
plt.title("AHR999：Time-warp + 带权缩放 拟合（含峰后）")
plt.legend(); plt.tight_layout()
plt.savefig(OUTDIR / "ahr999_fit_timewarp_scaled.png", dpi=170); plt.close()

print("已输出到：", OUTDIR.resolve())


C:\Users\nickc\AppData\Local\Temp\ipykernel_7504\2701506706.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
C:\Users\nickc\AppData\Local\Temp\ipykernel_7504\2701506706.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")


已输出到： C:\Users\nickc\yanda\btc\btc_fit_outputs09142025


: 